# F10 - Metrics, overfitting, underfitting, data leakage, and cross-validation

**Hardware:** CPU-only · **Time:** 3-4 hours · **Prerequisite:** F09

This notebook makes evaluation visible. It uses an imbalanced synthetic target,
shows why accuracy can mislead, plots the overfitting curve of a decision tree,
and demonstrates target leakage that produces a perfect but useless model.

In [1]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
print("pandas", pd.__version__)

pandas 2.3.3


## 1. An imbalanced target

`at_risk` is 1 when the final score is below 70. Only about one in five students
is positive, so a model that always predicts "not at risk" already scores about
0.79 accuracy. Accuracy alone would call that model good. It is not.

In [2]:
def make_students(n: int = 500, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    study = rng.gamma(2.0, 2.0, n).clip(0, 20)
    attendance = rng.uniform(40, 100, n)
    prior = rng.normal(65, 12, n).clip(0, 100)
    sleep = rng.normal(7, 1.2, n).clip(3, 10)
    tutoring = rng.choice(["yes", "no"], n, p=[0.35, 0.65])
    course = rng.choice(["Math", "Physics", "CS"], n, p=[0.4, 0.3, 0.3])
    final_score = (
        15 + 2.6 * study + 0.30 * attendance + 0.40 * prior + 0.8 * sleep
        + 4.0 * (tutoring == "yes")
        + 3.0 * pd.Series(course).map({"Math": 0.2, "Physics": 0.0, "CS": 0.45}).to_numpy()
        + rng.normal(0, 5, n)
    ).clip(0, 100)
    return pd.DataFrame(
        {
            "study_hours": study.round(2),
            "attendance": attendance.round(1),
            "prior_score": prior.round(1),
            "sleep_hours": sleep.round(2),
            "tutoring": tutoring,
            "course": course,
            "final_score": final_score.round(1),
        }
    )


students = make_students()
students["at_risk"] = (students["final_score"] < 70).astype(int)
X = pd.get_dummies(students[["study_hours", "attendance", "prior_score", "sleep_hours", "tutoring", "course"]], drop_first=True, dtype=float)
y = students["at_risk"].to_numpy()
print("positive rate:", round(y.mean(), 3))
print("majority accuracy:", round(max(y.mean(), 1 - y.mean()), 3))
assert 0.1 < y.mean() < 0.35

positive rate: 0.212
majority accuracy: 0.788


## 2. Confusion matrix and threshold-dependent metrics

The classifier returns a probability. A threshold turns it into a label. Lower
the threshold to catch more at-risk students (higher recall, lower precision);
raise it to reduce false alarms.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
scaler = StandardScaler().fit(X_train)
model = LogisticRegression(max_iter=2000, random_state=SEED).fit(scaler.transform(X_train), y_train)
probability = model.predict_proba(scaler.transform(X_test))[:, 1]
pred = (probability >= 0.5).astype(int)
print("confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, pred))
print("accuracy:", round(accuracy_score(y_test, pred), 3))
print("precision:", round(precision_score(y_test, pred), 3))
print("recall:", round(recall_score(y_test, pred), 3))
print("F1:", round(f1_score(y_test, pred), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, probability), 3))
print("PR-AUC:", round(average_precision_score(y_test, probability), 3))
assert roc_auc_score(y_test, probability) > 0.8

confusion matrix [[TN, FP], [FN, TP]]:
[[114   4]
 [  8  24]]
accuracy: 0.92
precision: 0.857
recall: 0.75
F1: 0.8
ROC-AUC: 0.963
PR-AUC: 0.892


## 3. Choose a threshold from costs, not habit

Suppose a missed at-risk student is twice as costly as a false alarm. The table
shows the trade-off; there is no universally correct threshold. Document the
cost assumption and the chosen value.

In [4]:
rows = []
for threshold in [0.3, 0.4, 0.5, 0.6]:
    labels = (probability >= threshold).astype(int)
    rows.append(
        {
            "threshold": threshold,
            "precision": precision_score(y_test, labels, zero_division=0),
            "recall": recall_score(y_test, labels, zero_division=0),
            "f1": f1_score(y_test, labels, zero_division=0),
        }
    )
thresholds = pd.DataFrame(rows)
print(thresholds.round(3).to_string(index=False))
assert thresholds.loc[thresholds["threshold"] == 0.3, "recall"].iloc[0] >= thresholds.loc[thresholds["threshold"] == 0.6, "recall"].iloc[0]

 threshold  precision  recall    f1
       0.3      0.763   0.906 0.829
       0.4      0.788   0.812 0.800
       0.5      0.857   0.750 0.800
       0.6      0.909   0.625 0.741


## 4. Cross-validation gives a mean and a spread

A single split is one sample from a noisy process. Stratified 5-fold
cross-validation preserves the class ratio in every fold and reports fold-to-fold
variation. Report both mean and standard deviation.

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_model = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=SEED)),
    ]
)
scores = cross_val_score(cv_model, X, y, cv=cv, scoring="average_precision")
print("fold PR-AUC:", np.round(scores, 3))
print("mean:", round(scores.mean(), 3), "std:", round(scores.std(), 3))
assert scores.mean() > 0.6

fold PR-AUC: [0.736 0.872 0.898 0.868 0.885]
mean: 0.852 std: 0.059


## 5. Overfitting by increasing model complexity

A decision tree with depth 1 underfits. As depth increases, training accuracy
approaches 1.0 while validation accuracy peaks and then falls. The best depth is
the one selected on validation data, not the one with perfect training accuracy.

In [6]:
depths = list(range(1, 16))
train_scores, validation_scores = [], []
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.30, random_state=SEED, stratify=y)
for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=SEED).fit(X_tr, y_tr)
    train_scores.append(tree.score(X_tr, y_tr))
    validation_scores.append(tree.score(X_va, y_va))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(depths, train_scores, marker="o", label="training accuracy")
ax.plot(depths, validation_scores, marker="s", label="validation accuracy")
ax.set_xlabel("Maximum tree depth")
ax.set_ylabel("Accuracy")
ax.set_title("Overfitting as tree depth increases")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "f10_overfitting_curve.png", bbox_inches="tight")
plt.close(fig)
best_depth = depths[int(np.argmax(validation_scores))]
print("best validation depth:", best_depth)
print("train at depth 15:", round(train_scores[-1], 3), "| validation:", round(validation_scores[-1], 3))
assert train_scores[-1] > validation_scores[-1]
assert (OUTPUT_DIR / "f10_overfitting_curve.png").exists()

best validation depth: 4
train at depth 15: 1.0 | validation: 0.82


## 6. Target leakage produces a perfect lie

Add `final_score` as a feature. It determines `at_risk` almost exactly, so the
model scores near 1.0. In a real deployment the final score is unavailable when
the prediction must be made. Remove it and report the honest score.

In [7]:
X_leaky = X.copy()
X_leaky["final_score"] = students["final_score"].to_numpy()
Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leaky, y, test_size=0.30, random_state=SEED, stratify=y
)
leaky_model = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(Xl_train, yl_train)
leaky_accuracy = leaky_model.score(Xl_test, yl_test)

Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
honest_model = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(Xh_train, yh_train)
honest_accuracy = honest_model.score(Xh_test, yh_test)
print("accuracy with final_score:", round(leaky_accuracy, 3))
print("accuracy without final_score:", round(honest_accuracy, 3))
assert leaky_accuracy > honest_accuracy

accuracy with final_score: 1.0
accuracy without final_score: 0.86


## Check your work

- [ ] You can explain why accuracy is misleading for an imbalanced target.
- [ ] You reported precision, recall, F1, ROC-AUC, and PR-AUC.
- [ ] You chose a threshold and stated the cost assumption.
- [ ] You can identify the overfitting region in the depth curve.
- [ ] You can explain why `final_score` is leakage for this prediction task.

**Next:** F11 - Neural network fundamentals.